# Notebook 02: Data Cleaning & Normalization

Reads from `raw_transactions`, applies data quality rules, derives category labels, and populates the normalized `customers`, `products`, `orders`, `order_items` tables.

In [ ]:
# ── Imports & DB Connection ────────────────────────────
import os
import re

import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# Update DB_PASSWORD before running
DB_USER     = 'root'
DB_PASSWORD = 'your_password'   # ← Update DB_PASSWORD before running
DB_HOST     = 'localhost'
DB_PORT     = 3306
DB_NAME     = 'ecommerce_clv'

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4',
    echo=False
)

print('Connected ✓')

In [ ]:
# ── Load Raw Data from MySQL ───────────────────────────
df = pd.read_sql('SELECT * FROM raw_transactions', engine)

print('Shape:', df.shape)

In [ ]:
# ── Data Quality Audit ────────────────────────────────
print('## Data Quality Audit')
print()

total_rows = len(df)

null_customer_id = df['CustomerID'].isna().sum()

cancelled_invoices = df['InvoiceNo'].astype(str).str.startswith('C').sum()

zero_neg_price = (df['UnitPrice'] <= 0).sum()

NON_MERCH_EXACT = {
    'POST', 'DOT', 'M', 'C2', 'D', 'BANK CHARGES',
    'PADS', 'AMAZONFEE', 'CRUK', 'S', 'B', 'DCGSSGIRL', 'DCGSSBOY'
}
non_merch_mask = (
    df['StockCode'].astype(str).isin(NON_MERCH_EXACT) |
    df['StockCode'].astype(str).str.startswith('gift_') |
    df['StockCode'].astype(str).str.startswith('DCGS')
)
non_merch_rows = non_merch_mask.sum()

zero_neg_qty = (df['Quantity'] <= 0).sum()

print(f'Total raw rows:              {total_rows:>10,}')
print(f'Null CustomerID:             {null_customer_id:>10,}  (expect ~135,080)')
print(f'Cancelled invoices (C...):   {cancelled_invoices:>10,}  (expect ~9,288 in raw)')
print(f'Zero or negative UnitPrice:  {zero_neg_price:>10,}')
print(f'Non-merchandise StockCodes:  {non_merch_rows:>10,}')
print(f'Zero or negative Quantity:   {zero_neg_qty:>10,}')
print()

summary = pd.DataFrame({
    'Exclusion Rule': [
        'Null CustomerID',
        'Cancelled invoices (InvoiceNo starts C)',
        'Zero or negative UnitPrice',
        'Non-merchandise StockCodes',
        'Zero or negative Quantity',
    ],
    'Row Count': [
        null_customer_id,
        cancelled_invoices,
        zero_neg_price,
        non_merch_rows,
        zero_neg_qty,
    ]
})

print('Data Quality Summary:')
print(summary.to_string(index=False))

In [ ]:
# ── Cleaning Pipeline ─────────────────────────────────
df_clean = df.copy()
print(f'Starting rows: {len(df_clean):,}')

# Step 1: Remove non-merchandise StockCodes
NON_MERCH = {
    'POST', 'DOT', 'M', 'C2', 'D', 'BANK CHARGES',
    'PADS', 'AMAZONFEE', 'CRUK', 'S', 'B', 'DCGSSGIRL', 'DCGSSBOY'
}
non_merch_filter = lambda code: (
    str(code) in NON_MERCH or
    str(code).startswith('gift_') or
    str(code).startswith('DCGS')
)
df_clean = df_clean[~df_clean['StockCode'].astype(str).map(non_merch_filter)]
print(f'After removing non-merchandise: {len(df_clean):,}')

# Step 2: Remove cancelled invoices
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f'After removing cancellations:   {len(df_clean):,}')

# Step 3: Remove null CustomerID
df_clean = df_clean[df_clean['CustomerID'].notna()]
print(f'After removing null CustomerID: {len(df_clean):,}')

# Step 4: Remove zero/negative UnitPrice
df_clean = df_clean[df_clean['UnitPrice'] > 0]
print(f'After removing bad UnitPrice:   {len(df_clean):,}')

# Step 5: Remove zero/negative Quantity
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f'After removing bad Quantity:    {len(df_clean):,}')

# Normalize column names to snake_case
df_clean = df_clean.rename(columns={
    'InvoiceNo':   'invoice_no',
    'StockCode':   'stock_code',
    'Description': 'description',
    'Quantity':    'quantity',
    'InvoiceDate': 'invoice_date',
    'UnitPrice':   'unit_price',
    'CustomerID':  'customer_id',
    'Country':     'country',
})

df_clean['invoice_date'] = pd.to_datetime(df_clean['invoice_date'])
df_clean['customer_id']  = df_clean['customer_id'].astype(str).str.strip()
df_clean['description']  = df_clean['description'].astype(str).str.strip().str.upper()

print(f'\nFinal clean rows: {len(df_clean):,}')

In [ ]:
# ── Category Derivation ───────────────────────────────
CATEGORY_RULES = [
    ('Christmas & Seasonal',  r'CHRISTMAS|XMAS|ADVENT|EASTER|HALLOWEEN|WINTER WONDERLAND'),
    ('Bags & Accessories',    r'\bBAG\b|PURSE|WALLET|JEWELLERY|NECKLACE|BRACELET|UMBRELLA|\bHANDBAG'),
    ('Kitchen & Dining',      r'MUG|\bCUP\b|PLATE|BOWL|JAM\b|CAKE|BAKING|KITCHEN|TEA TOWEL|\bSPOON\b|JAR|KETTLE|SAUCER|\bJUG\b|\bTEACUP\b|TEAPOT|SUGAR|BISCUIT TIN|BUTTER DISH'),
    ('Home Decor',            r'\bLIGHT\b|CANDLE|HOLDER|DECORATION|FRAME|MIRROR|CLOCK|LANTERN|\bSIGN\b|DOORMAT|CUSHION|BLANKET|WALLPAPER|BUNTING|WREATH'),
    ('Stationery & Gifts',    r'\bCARD\b|NOTEBOOK|\bPEN\b|\bGIFT\b|WRAP|PAPER|STICKER|\bPENCIL\b|CALENDAR|DIARY|BOOKMARK|PENCIL CASE'),
    ('Toys & Games',          r'\bTOY\b|GAME|PUZZLE|\bDOLL\b|RATTLE|BALL\b|SPINNING'),
    ('Garden & Outdoor',      r'GARDEN|PLANT POT|OUTDOOR|WATERING|BIRD FEEDER|PLANTER'),
    ('Bath & Body',           r'\bSOAP\b|\bBATH\b|\bTOWEL\b|SPONGE|LOOFAH'),
]

def categorize(description):
    """Assign a category based on keyword rules; fallback to 'Other'."""
    desc = str(description).upper()
    for category, pattern in CATEGORY_RULES:
        if re.search(pattern, desc):
            return category
    return 'Other'

df_clean['category'] = df_clean['description'].apply(categorize)

print('Category distribution:')
print(df_clean['category'].value_counts())

In [ ]:
# ── Build & Load: customers ───────────────────────────
customers_df = df_clean.groupby('customer_id').agg(
    country=('country', 'first'),
    first_purchase_date=('invoice_date', 'min'),
    last_purchase_date=('invoice_date', 'max'),
).reset_index()

customers_df['is_active'] = 1

customers_df.to_sql('customers', engine, if_exists='append', index=False)
customers_df.to_csv('../data/processed/customers.csv', index=False)

print(f'✅ Customers loaded: {len(customers_df):,}')

In [ ]:
# ── Build & Load: products ────────────────────────────
# One row per stock_code: most-frequent description, mode category, mean unit_price
products_df = (
    df_clean
    .groupby('stock_code')
    .agg(
        description=('description', lambda x: x.value_counts().idxmax()),
        category=('category',    lambda x: x.value_counts().idxmax()),
        unit_price=('unit_price', 'mean'),
    )
    .reset_index()
)
products_df['unit_price'] = products_df['unit_price'].round(2)

products_df.to_sql('products', engine, if_exists='append', index=False)
products_df.to_csv('../data/processed/products.csv', index=False)

print(f'✅ Products loaded: {len(products_df):,}')

In [ ]:
# ── Build & Load: orders ──────────────────────────────
orders_df = (
    df_clean[['invoice_no', 'customer_id', 'invoice_date', 'country']]
    .drop_duplicates(subset='invoice_no')
    .copy()
)
orders_df['order_status'] = 'Completed'
orders_df = orders_df.rename(columns={'invoice_date': 'order_date'})

orders_df.to_sql('orders', engine, if_exists='append', index=False)
orders_df.to_csv('../data/processed/orders.csv', index=False)

print(f'✅ Orders loaded: {len(orders_df):,}')

In [ ]:
# ── Build & Load: order_items ─────────────────────────
order_items_df = df_clean[['invoice_no', 'stock_code', 'quantity', 'unit_price']].copy()

order_items_df.to_sql(
    'order_items', engine,
    if_exists='append',
    index=False,
    chunksize=5000,
    method='multi'
)
order_items_df.to_csv('../data/processed/order_items.csv', index=False)

print(f'✅ Order items loaded: {len(order_items_df):,}')

In [ ]:
# ── Verification ──────────────────────────────────────
table_counts = {}
for tbl in ['customers', 'products', 'orders', 'order_items']:
    result = pd.read_sql(f'SELECT COUNT(*) as row_count FROM {tbl}', engine)
    table_counts[tbl] = result['row_count'].iloc[0]

verification_df = pd.DataFrame({
    'Table':     list(table_counts.keys()),
    'Row Count': [f"{v:,}" for v in table_counts.values()],
})

print('Table row counts:')
print(verification_df.to_string(index=False))
print()
print('✅ All tables populated and verified.')

## Summary: Data Quality Decisions

The following cleaning rules were applied to the raw UCI Online Retail data:

| Rule | Rationale |
|------|-----------|
| Removed null `CustomerID` (~135,080 rows) | Cannot attribute revenue to a customer; unusable for CLV |
| Removed cancelled invoices (`InvoiceNo` starts with `C`) | Cancellations represent reversed transactions, not realized revenue |
| Removed non-merchandise StockCodes (`POST`, `DOT`, `M`, `C2`, `D`, etc.) | These are postage, adjustments, and bank charges — not product sales |
| Removed `UnitPrice <= 0` | Zero/negative prices indicate test records or manual adjustments |
| Removed `Quantity <= 0` | Negative quantities are returns, already handled by cancellation filter |

**Category labels** were derived from product descriptions using regex pattern matching across 8 categories.
Unmatched descriptions fall into `'Other'`.

**Normalized tables populated:**
- `customers` — one row per unique `CustomerID`, with country and purchase date range
- `products` — one row per `StockCode`, with modal description, category, and mean price
- `orders` — one row per `InvoiceNo`, all status set to `'Completed'` (cancellations excluded)
- `order_items` — line-level detail: invoice, stock code, quantity, unit price

**Next step:** Run `03_eda_analysis.ipynb` for exploratory data analysis.